In [0]:

from pyspark.sql.functions import first, rand, count
dates = spark.sql("SELECT explode(sequence(DATE'2024-01-01', DATE'2024-03-24', INTERVAL 1 DAY)) as  calendar_date")
c_id = spark.sql("SELECT explode(sequence(1,100, 1)) as  client_id")
types = spark.sql("""SELECT concat("col_", colName) as col_name from (SELECT explode(sequence(1,7000, 1)) as  colName)""")
 
dates = dates.repartition(100000)
c_id = c_id.repartition(110000)
types = types.repartition(100000)
 
# df_cartesian = c_id.crossJoin(types.select("col_name")).select("client_id","col_name")
df_cartesian = c_id.crossJoin(dates.select("calendar_date")).crossJoin(types.select("col_name")).select("client_id","calendar_date","col_name")
# df_cartesian2 = df_cartesian.groupBy("calendar_date").agg(count("client_id"))
 
# display(df_cartesian2.limit(1000))
 
df_cartesian = df_cartesian.withColumn("val", (rand()*10).cast("int"))


In [0]:
display(df_cartesian.select("client_id").distinct())

In [0]:
from pyspark.sql.functions import first
 
df_grp = df_cartesian.groupBy("col_name").pivot("col_name").agg((first("val").alias("val")))
# df_grp.write.saveAsTable("demo_catalog.wine_db.wide_table3")
 
#split_weights = [1.0] * 1000
#splits = df_cartesian.randomSplit(split_weights)
 
#for df_split in splits:
#    df_grp = df_split.groupBy("client_id","calendar_date").pivot("col_name").agg((first("val").alias("val")))
#    df_split.write.saveAsTable("demo_catalog.wine_db.wide_table")
 
 
#df_grp_part.write.saveAsTable("demo_catalog.wine_db.wide_table")
display(df_grp)

In [0]:
from pyspark.sql.functions import first, rand, lit, col, expr
from pyspark.sql import Window
import random

# Step 1: Generate 7000 row IDs
row_ids = spark.sql("SELECT explode(sequence(1, 70, 1)) as row_id")

# Step 2: Generate a smaller number of column names that we'll pivot
# Using 100 column names and 70 values per column to reach 7000 columns total
col_base = spark.sql("SELECT explode(sequence(1, 100, 1)) as col_base")
col_suffix = spark.sql("SELECT explode(sequence(1, 70, 1)) as col_suffix")
column_names = col_base.crossJoin(col_suffix).withColumn(
    "col_name", 
    expr("concat('col_', cast(col_base as string), '_', cast(col_suffix as string))")
).select("col_name")

# Step 3: Create a dataset with row_id, col_name, and random values
df_base = row_ids.crossJoin(column_names).withColumn(
    "value", 
    rand(seed=42) * 100  # Random values between 0 and 100
)

# Step 4: Pivot the data to get 7000 columns
# We'll do this by processing batches of columns to avoid memory issues
def pivot_batch(batch_df, batch_cols):
    return batch_df.groupBy("row_id").pivot("col_name", batch_cols).agg(first("value"))

# Process in batches of columns
batch_size = 10
column_list = [row.col_name for row in column_names.collect()]
# result_df = None

In [0]:
df = pivot_batch(df_base, column_list)
display(df)